# Phase 4.3 — Calibration Verification

7 PD sources:
1. LightGBM tuned + Platt (PRIMARY)
2. LR full-F6E + Platt
3. LR no-F6E + Platt
4. Scorecard no-F6E + Platt
5. Stressed Gini 0.30 (perturbed from #1)
6. Stressed Gini 0.45
7. Stressed Gini 0.60

Note: scorecard full-F6E robustness exists as metrics only (no row-level PD
parquet was generated in Phase 2B optional run); excluded from this round.
Per source: ECE, Brier, mean predicted vs observed, O:E by decile,
reliability data — saved to `artifacts/calibration_verification/`.

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation import compute_gini, compute_ks, compute_brier, compute_calibration_metrics

ECO_PATH = REPO_ROOT / "artifacts/economic_framework/economics_per_account.parquet"
PDV_PATH = REPO_ROOT / "artifacts/pd_quality_stress/pd_variants.parquet"
OUT_DIR  = REPO_ROOT / "artifacts/calibration_verification"
OUT_DIR.mkdir(parents=True, exist_ok=True)

eco = pd.read_parquet(ECO_PATH)
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
print(f"OOT rows: {len(oot):,}, base default rate: {(oot['default_flag_12m']==1).mean():.5%}")

pdv = pd.read_parquet(PDV_PATH)
pdv = pdv.set_index("aid").loc[oot["aid"]].reset_index()  # align order
print(f"PD variants aligned: {pdv.shape}")

T0 = time.time()

OOT rows: 64,027, base default rate: 1.91638%
PD variants aligned: (64027, 8)


In [2]:
PD_SOURCES = {
    "LightGBM Platt (PRIMARY)":     oot["pd_lgb_platt"].to_numpy(),
    "LR full-F6E Platt":            oot["pd_lr_full_platt"].to_numpy(),
    "LR no-F6E Platt":              oot["pd_lr_nof6e_platt"].to_numpy(),
    "Scorecard no-F6E Platt":       oot["pd_sc_platt"].to_numpy(),
    "Stressed Gini 0.60":           pdv["pd_gini_60"].to_numpy(),
    "Stressed Gini 0.45":           pdv["pd_gini_45"].to_numpy(),
    "Stressed Gini 0.30":           pdv["pd_gini_30"].to_numpy(),
}
y = oot["default_flag_12m"].astype(int).to_numpy()
base_rate = float(y.mean())
print(f"Base rate: {base_rate:.5%}")

Base rate: 1.91638%


In [3]:
# Compute per-source metrics
def reliability_table(y_true, y_pred, n_bins=10):
    s = pd.Series(y_pred)
    cuts = pd.qcut(s.rank(method="first"), n_bins, labels=False, duplicates="drop")
    rows = []
    for b in range(int(cuts.max()) + 1):
        m = cuts == b
        if not m.any(): continue
        rows.append({
            "bin": b + 1,
            "n": int(m.sum()),
            "mean_pred": float(y_pred[m].mean()),
            "obs_rate": float(y_true[m].mean()),
            "obs_to_exp": (float(y_true[m].mean()) / float(y_pred[m].mean())) if y_pred[m].mean() > 0 else float("nan"),
        })
    return pd.DataFrame(rows)

summary_rows = []
reliability_data = {}
oe_data = {}
for name, p in PD_SOURCES.items():
    p = np.clip(p, 1e-9, 1 - 1e-9)
    cal = compute_calibration_metrics(y, p, n_bins=10)
    summary_rows.append({
        "pd_source": name,
        "n": len(y),
        "base_rate": base_rate,
        "mean_predicted": float(p.mean()),
        "AUC": (compute_gini(y, p) + 1) / 2,
        "Gini": compute_gini(y, p),
        "KS": compute_ks(y, p),
        "Brier": compute_brier(y, p),
        "ECE": cal.get("ece", float("nan")),
        "mean_pred_to_obs_ratio": float(p.mean()) / base_rate if base_rate > 0 else float("nan"),
    })
    rel = reliability_table(y, p)
    reliability_data[name] = rel
    oe_data[name] = {f"o_to_e_bin{i}": float(rel.iloc[i-1]["obs_to_exp"]) if i <= len(rel) else float("nan")
                     for i in range(1, 11)}

summary = pd.DataFrame(summary_rows)
print("=== CALIBRATION SUMMARY ===")
print(summary.round(5).to_string(index=False))

# Save
summary.to_csv(OUT_DIR / "calibration_summary.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'calibration_summary.csv'}")

oe_df = pd.DataFrame(oe_data).T
oe_df.to_csv(OUT_DIR / "o_to_e_by_decile.csv")
print(f"Saved: {OUT_DIR / 'o_to_e_by_decile.csv'}")

# Save reliability data per source
all_rel = []
for name, rel in reliability_data.items():
    rel = rel.copy()
    rel.insert(0, "pd_source", name)
    all_rel.append(rel)
all_rel_df = pd.concat(all_rel, ignore_index=True)
all_rel_df.to_csv(OUT_DIR / "reliability_data.csv", index=False)
print(f"Saved: {OUT_DIR / 'reliability_data.csv'}")

print(f"\nWall: {time.time()-T0:.1f}s")

=== CALIBRATION SUMMARY ===
               pd_source     n  base_rate  mean_predicted     AUC    Gini      KS   Brier     ECE  mean_pred_to_obs_ratio
LightGBM Platt (PRIMARY) 64027    0.01916         0.00939 0.90222 0.80443 0.64611 0.01721 0.00980                 0.48977
       LR full-F6E Platt 64027    0.01916         0.02070 0.86404 0.72808 0.55945 0.01671 0.00261                 1.08024
         LR no-F6E Platt 64027    0.01916         0.02076 0.83998 0.67997 0.54579 0.01703 0.00294                 1.08304
  Scorecard no-F6E Platt 64027    0.01916         0.00918 0.80304 0.60607 0.48131 0.01794 0.00998                 0.47900
      Stressed Gini 0.60 64027    0.01916         0.01916 0.80148 0.60296 0.46904 0.02039 0.01429                 1.00000
      Stressed Gini 0.45 64027    0.01916         0.01916 0.72279 0.44559 0.32829 0.02418 0.02381                 1.00000
      Stressed Gini 0.30 64027    0.01916         0.02177 0.65134 0.30268 0.22881 0.02868 0.03124                 1.13